# Multitask D-Optimal Design Plots

This notebook reads the CSV files produced by `06 Multitask D_optimal Design.ipynb` and generates all matplotlib plots.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

stochastic_reference_history_csv = Path("stochastic_reference_history.csv")
gamma_sweep_csv = Path("gamma_sweep_results_vertex.csv")
gamma_sweep_worst_case_gap_csv = Path("gamma_sweep_worst_case_gaps_vertex.csv")
adaptive_gamma_csv = Path("adaptive_gamma_results_vertex.csv")
adaptive_gamma_worst_case_gap_csv = Path("adaptive_gamma_worst_case_gaps_vertex.csv")
theta_reference_csv = Path("theta_mean_lambda_reference_results.csv")
lambda_point_comparison_csv = Path("lambda_point_comparison.csv")

# Plot parameters.
# `plot_gamma_indices` indexes the fixed-gamma values after sorting them from small to large.
# Set to None for all fixed gammas, an int for one gamma, or a list such as [0, 3, -1].
# plot_gamma_indices = [0, 5, 7, 10]
# plot_gamma_indices = [10, 12, 14, 15]
plot_gamma_indices = None
# plot_gamma_indices = [0 ,1, 2, 3]
plot_gammas = [1.0, 0.1, 0.01, 0.001]  # Optional explicit override; usually leave this as None and use indices above.
max_plot_iteration = 500  # Example: 200. Set to None to plot all saved iterations.
include_adaptive = True

smooth_window = 20
log_plot_font_size = 55

required_csvs = [
    stochastic_reference_history_csv,
    gamma_sweep_csv,
    gamma_sweep_worst_case_gap_csv,
    adaptive_gamma_csv,
    theta_reference_csv,
]
if include_adaptive:
    required_csvs.append(adaptive_gamma_worst_case_gap_csv)
missing_csvs = [str(path) for path in required_csvs if not path.exists()]
if missing_csvs:
    raise FileNotFoundError(
        "Run `06 Multitask D_optimal Design.ipynb` first. Missing CSV files: " + ", ".join(missing_csvs)
    )

font_size = 55
plot_figsize = (20, 18)
plot_linewidth = 10
plot_markersize = 10

accessible_colors = ["#B30000", "#006600", "#0000FF", "#990099", "#000000"]
fixed_gamma_marker = "D"
adaptive_gamma_marker = "^"
fixed_gamma_colors = [color for color in accessible_colors if color.lower() != "#000000"]
fixed_gamma_linestyle = "-."

history_palette = {
    "objective_estimate": "#0000FF",
    "reg_objective_estimate": "#FF0000",
}


In [ ]:
history = pd.read_csv(stochastic_reference_history_csv)
gamma_sweep_results = pd.read_csv(gamma_sweep_csv)
gamma_sweep_worst_case_gaps = pd.read_csv(gamma_sweep_worst_case_gap_csv)
adaptive_gamma_results = pd.read_csv(adaptive_gamma_csv)
adaptive_gamma_worst_case_gaps = pd.read_csv(adaptive_gamma_worst_case_gap_csv) if include_adaptive else pd.DataFrame()
theta_reference_results = pd.read_csv(theta_reference_csv)

full_comparison_results = pd.concat([gamma_sweep_results, adaptive_gamma_results], ignore_index=True)
full_comparison_results = full_comparison_results.sort_values(["experiment", "t"]).copy()
full_worst_case_gap_results = pd.concat(
    [gamma_sweep_worst_case_gaps, adaptive_gamma_worst_case_gaps],
    ignore_index=True,
)

available_fixed_gammas = [
    float(gamma)
    for gamma in sorted(
        full_comparison_results.loc[full_comparison_results["mode"].eq("fixed"), "gamma"].dropna().unique()
    )
]
available_gamma_table = pd.DataFrame(
    {"gamma_index": range(len(available_fixed_gammas)), "gamma": available_fixed_gammas}
)


def resolve_plot_gammas(explicit_gammas=None, gamma_indices=None):
    if explicit_gammas is not None:
        return [float(gamma) for gamma in explicit_gammas]

    if gamma_indices is None:
        return None

    if np.isscalar(gamma_indices):
        indices = [int(gamma_indices)]
    else:
        indices = [int(index) for index in gamma_indices]

    resolved_gammas = []
    n_gammas = len(available_fixed_gammas)
    for index in indices:
        if index < -n_gammas or index >= n_gammas:
            raise IndexError(
                f"Gamma index {index} is out of range for {n_gammas} available fixed-gamma values. "
                "Use `available_gamma_table` to inspect valid indices."
            )
        resolved_gammas.append(available_fixed_gammas[index])
    return resolved_gammas


resolved_plot_gammas = resolve_plot_gammas(
    explicit_gammas=plot_gammas,
    gamma_indices=plot_gamma_indices,
)


def filter_comparison_results(df, gammas=None, max_iteration=None, include_adaptive=True):
    filtered_parts = []
    fixed_results = df[df["mode"].eq("fixed")].copy()

    if gammas is not None:
        gamma_values = np.array([float(gamma) for gamma in gammas], dtype=float)
        gamma_mask = np.zeros(len(fixed_results), dtype=bool)
        for gamma_value in gamma_values:
            gamma_mask |= np.isclose(fixed_results["gamma"].to_numpy(dtype=float), gamma_value)
        fixed_results = fixed_results[gamma_mask].copy()

    filtered_parts.append(fixed_results)

    if include_adaptive:
        filtered_parts.append(df[df["experiment"].eq("adaptive gamma")].copy())

    filtered = pd.concat(filtered_parts, ignore_index=True)

    if max_iteration is not None:
        filtered = filtered[filtered["t"] <= int(max_iteration)].copy()

    filtered = filtered.sort_values(["experiment", "t"]).copy()
    if filtered.empty:
        raise ValueError(
            "No experiment histories match `plot_gamma_indices`, `plot_gammas`, and `max_plot_iteration`. "
            "Use `available_gamma_table` to inspect available fixed-gamma values."
        )
    return filtered


comparison_results = filter_comparison_results(
    full_comparison_results,
    gammas=resolved_plot_gammas,
    max_iteration=max_plot_iteration,
    include_adaptive=include_adaptive,
)
comparison_results_no_adaptive = comparison_results[
    comparison_results["experiment"] != "adaptive gamma"
].copy()

if resolved_plot_gammas is None:
    focused_gamma = available_fixed_gammas[0]
elif len(resolved_plot_gammas) > 0:
    focused_gamma = resolved_plot_gammas[0]
else:
    focused_gamma = available_fixed_gammas[0]

single_gamma_results = filter_comparison_results(
    full_comparison_results,
    gammas=[focused_gamma],
    max_iteration=max_plot_iteration,
    include_adaptive=False,
)


def filter_worst_case_gap_results(df, gammas=None, max_iteration=None, include_adaptive=True):
    filtered_parts = []
    fixed_results = df[df["mode"].eq("fixed")].copy()

    if gammas is not None:
        gamma_values = np.array([float(gamma) for gamma in gammas], dtype=float)
        gamma_mask = np.zeros(len(fixed_results), dtype=bool)
        for gamma_value in gamma_values:
            gamma_mask |= np.isclose(fixed_results["gamma"].to_numpy(dtype=float), gamma_value)
        fixed_results = fixed_results[gamma_mask].copy()

    filtered_parts.append(fixed_results)

    if include_adaptive:
        filtered_parts.append(df[df["experiment"].eq("adaptive gamma")].copy())

    filtered = pd.concat(filtered_parts, ignore_index=True)

    if max_iteration is not None:
        filtered = filtered[filtered["epoch_index"] <= int(max_iteration)].copy()

    if filtered.empty:
        raise ValueError(
            "No worst-case gap rows match `plot_gamma_indices`, `plot_gammas`, `include_adaptive`, and `max_plot_iteration`. "
            "Use `available_gamma_table` to inspect available fixed-gamma values."
        )

    filtered = filtered.rename(columns={"epoch_index": "t"})
    return filtered.sort_values(["experiment", "t"]).copy()


worst_case_gap_results = filter_worst_case_gap_results(
    full_worst_case_gap_results,
    gammas=resolved_plot_gammas,
    max_iteration=max_plot_iteration,
    include_adaptive=include_adaptive,
)


def keep_regular_adaptive_gap_snapshots(df):
    regular = df.copy()
    adaptive_mask = regular["experiment"].eq("adaptive gamma")
    if "snapshot_reason" not in regular.columns:
        return regular

    snapshot_reason = regular["snapshot_reason"].fillna("checkpoint").astype(str)
    keep_adaptive = snapshot_reason.isin(["initial", "checkpoint"])
    return regular[(~adaptive_mask) | keep_adaptive].copy()


worst_case_gap_results_regular_adaptive = keep_regular_adaptive_gap_snapshots(worst_case_gap_results)

theta_reference_level = float(theta_reference_results["mean_lambda_theta_objective"].min())

summary_tables = {
    "available_gamma_table": available_gamma_table,
    "history_tail": history.loc[:, ["t", "objective_estimate", "reg_objective_estimate"]].tail(10),
    "downstream_summary": comparison_results.groupby("experiment")["reg_objective_estimate"].agg(["first", "last", "min"]),
    "worst_case_gap_summary": worst_case_gap_results.groupby("experiment")["worst_case_gap"].agg(["first", "last", "max"]),
    "theta_reference_level": pd.DataFrame(
        {"reference_metric": ["mean_lambda_theta_objective"], "value": [theta_reference_level]}
    ),
}
available_gamma_table


In [ ]:
def finish_plot(ax, title, ylabel, xlabel="# gradient calls", legend=True):
    ax.set_title(title, fontsize=font_size)
    ax.set_xlabel(xlabel, fontsize=font_size)
    ax.set_ylabel(ylabel, fontsize=font_size)
    ax.set_xlim(left=0, right=max_plot_iteration)
    ax.tick_params(axis="both", which="both", labelsize=font_size)
    for tick_label in ax.get_xticklabels(which="both") + ax.get_yticklabels(which="both"):
        tick_label.set_fontsize(font_size)
    ax.yaxis.get_offset_text().set_fontsize(font_size)
    ax.xaxis.get_offset_text().set_fontsize(font_size)
    ax.grid(True)
    if legend:
        ax.legend(fontsize=font_size)
    plt.tight_layout()
    plt.show()


def add_adaptive_phase_lines(ax, df):
    adaptive_frame = df[df["experiment"] == "adaptive gamma"].sort_values("t")
    if "phase" not in adaptive_frame.columns or adaptive_frame.empty:
        return
    phase_starts = adaptive_frame.groupby("phase", dropna=True).first().reset_index()
    for _, row in phase_starts.iloc[1:].iterrows():
        ax.axvline(
            x=float(row["t"]),
            color="#000000",
            linestyle=":",
            # markevery=20,
            linewidth=6,
            alpha=0.95,
            zorder=0,
        )


def experiment_style(label, fixed_labels, gamma_value=None):
    if label == "adaptive gamma":
        return {
        "color": "#000000",
        "marker": adaptive_gamma_marker,
        "linestyle": "--",
        "zorder": 4,
        "markersize": 30,
        }
    index = fixed_labels.index(label) if label in fixed_labels else 0
    return {
        "color": fixed_gamma_colors[index % len(fixed_gamma_colors)],
        "marker": fixed_gamma_marker,
        "linestyle": fixed_gamma_linestyle,
        "zorder": 2,
    }


def plot_history_series(df, value_col, title, ylabel="objective value", label=None):
    fig, ax = plt.subplots(figsize=plot_figsize)
    ax.plot(
        df["t"],
        df[value_col],
        linestyle="--",
        marker="o" if value_col == "objective_estimate" else "s",
        linewidth=plot_linewidth,
        markersize=plot_markersize,
        markevery=20,
        alpha=0.8,
        color=history_palette.get(value_col),
        label=label or value_col,
    )
    finish_plot(ax, title=title, ylabel=ylabel)


def plot_experiment_comparison(
    df,
    value_col,
    title,
    ylabel="objective value",
    reference_level=None,
    reference_label="mean-lambda L-BFGS benchmark",
):
    fig, ax = plt.subplots(figsize=plot_figsize)
    add_adaptive_phase_lines(ax, df)
    all_labels = list(df["experiment"].dropna().unique())
    fixed_labels = [label for label in all_labels if label != "adaptive gamma"]
    ordered_labels = fixed_labels + (["adaptive gamma"] if "adaptive gamma" in all_labels else [])

    for label in ordered_labels:
        frame = df[df["experiment"] == label].sort_values("t")
        gamma_value = float(frame["gamma"].iloc[0]) if "gamma" in frame.columns else np.nan
        style = experiment_style(label, fixed_labels, gamma_value=gamma_value)
        ax.plot(
            frame["t"],
            frame[value_col],
            linestyle=style["linestyle"],
            marker=style["marker"],
            markevery=20,
            linewidth=plot_linewidth,
            markersize=style.get("markersize", plot_markersize),
            alpha=0.8,
            color=style["color"],
            label=label,
            zorder=style["zorder"],
        )

    if reference_level is not None:
        ax.axhline(
            y=reference_level,
            color="#000000",
            linestyle=":",
            linewidth=4,
            alpha=0.8,
            label=reference_label,
            zorder=1,
        )

    finish_plot(ax, title=title, ylabel=ylabel)


def plot_worst_case_gap_comparison(
    df,
    title="Worst-Case Gap",
    ylabel="worst-case gap",
    phase_line_df=None,
    show_phase_lines=True,
):
    fig, ax = plt.subplots(figsize=plot_figsize)
    if show_phase_lines:
        add_adaptive_phase_lines(ax, df if phase_line_df is None else phase_line_df)
    all_labels = list(df["experiment"].dropna().unique())
    fixed_labels = [label for label in all_labels if label != "adaptive gamma"]
    ordered_labels = fixed_labels + (["adaptive gamma"] if "adaptive gamma" in all_labels else [])

    for label in ordered_labels:
        frame = df[df["experiment"] == label].sort_values("t")
        gamma_value = float(frame["gamma"].iloc[0]) if "gamma" in frame.columns else np.nan
        style = experiment_style(label, fixed_labels, gamma_value=gamma_value)
        ax.plot(
            frame["t"],
            frame["worst_case_gap"],
            linestyle=style["linestyle"],
            marker=style["marker"],
            markevery=1,
            linewidth=plot_linewidth,
            markersize=style.get("markersize", plot_markersize),
            alpha=0.8,
            color=style["color"],
            label=label,
            zorder=style["zorder"],
        )

    finish_plot(ax, title=title, ylabel=ylabel)


## Stochastic Reference Descent


In [ ]:
plot_history_series(
    history,
    value_col="objective_estimate",
    title="Objective Estimate",
    ylabel="objective value",
)


## Fixed and Adaptive Gamma Comparisons


In [ ]:
plot_experiment_comparison(
    comparison_results,
    value_col="reg_objective_estimate",
    title=None,
    ylabel="regularized objective",
)


In [ ]:
plot_experiment_comparison(
    comparison_results,
    value_col="objective_estimate",
    title=None,
    ylabel="unregularized objective",
)


In [ ]:
plot_experiment_comparison(
    single_gamma_results,
    value_col="objective_estimate",
    title=f"Unregularized Objective, Fixed Gamma = {focused_gamma:g}",
    ylabel="unregularized objective",
)


In [ ]:
plot_experiment_comparison(
    single_gamma_results,
    value_col="reg_objective_estimate",
    title=f"Regularized Objective, Fixed Gamma = {focused_gamma:g}",
    ylabel="regularized objective",
)


## Worst-Case Gap


In [ ]:
plot_worst_case_gap_comparison(
    worst_case_gap_results,
    title=None,
    ylabel="worst-case gap",
    phase_line_df=comparison_results,
)


In [ ]:
plot_worst_case_gap_comparison(
    worst_case_gap_results_regular_adaptive,
    title=None,
    ylabel="worst-case gap",
    show_phase_lines=False,
)


In [ ]:
plot_worst_case_gap_comparison(
    worst_case_gap_results,
    title=None,
    ylabel="worst-case gap",
    phase_line_df=comparison_results,
)
